# 03 · The Hero cascade — d8 global + EBM regime = the 0.12033 ceiling

*Edge-features arc · 01 discovery · 02 linear base · 03 hero cascade · 04 regime-MoE · 05 temporal-kNN  —  machinery: `ridge_pipeline_throughline.ipynb`*

**Verify first.** The deployed stack is a 3-stage cascade: `linbest` base + **d8 global XGB** (Hero A) +
**gated EBM regime** on the h16-19 close/AH leftover (Hero B). Machinery = `resid_amortized.preds_chunk`
(`resid_regime` arm, the through-line for ch. 04). Here we fold the cascade ladder + the small-sample law.

In [ ]:
import html, inspect, json, os, sys, textwrap
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

def find_repo(s):
    for q in [Path(s).resolve(), *Path(s).resolve().parents]:
        if (q / "resid_amortized.py").exists() and (q / "src").is_dir():
            return q
    raise FileNotFoundError("repo root")
REPO = find_repo(Path.cwd()); os.chdir(REPO); sys.path.insert(0, str(REPO))

def _details(f, open_=False):
    mod = f.__module__.replace("src.", "src/").replace(".", "/") + ".py"
    try:
        sig = ("class " + f.__name__) if inspect.isclass(f) else ("def " + f.__name__ + str(inspect.signature(f)))
    except (ValueError, TypeError):
        sig = f.__qualname__
    body = "```python\n" + textwrap.dedent(inspect.getsource(f)).rstrip() + "\n```"
    return (f"<details{' open' if open_ else ''}>\n<summary><code>{html.escape(mod + '  ·  ' + sig)}"
            f"</code></summary>\n\n{body}\n\n</details>")
def show_one(f):
    return Markdown(_details(f))
print("setup ok")

---
## 1 · Verify — the cascade ladder

In [ ]:
# cluster-established cascade QLIKE (writeup §1 on enetreg2, §8 the pre-DL dig on linbest)
ladder = pd.DataFrame([
    ("enetreg2 base",                      0.12314, "base (ch.02)"),
    ("+ XGB-d8 global (Hero A)",           0.12129, "global tree"),
    ("+ untuned EBM regime@h16-19 (Hero B)",0.12050, "regime stage"),
    ("+ path-shape ⊥ regime (deployed)",   0.12035, "deployed"),
    ("linbest + d8@cs0.5 global (Hero A')",0.12081, "no regime stage"),
    ("  + tuned XGB regime (pure power)",  0.12072, "XGB regime"),
    ("  + Optuna XGB regime (120-trial)",  0.12094, "XGB regime, proper search"),
    ("  + EBM regime  =  THE CEILING",     0.12033, "EBM regime"),
], columns=["stage", "qlike", "what"])
display(ladder)
print("Ceiling = 0.12033 (linbest + d8@cs0.5 + EBM regime). The MoE (ch.04) targets exactly this slot.")

## 2 · Interpret — the EBM regime is the *correct* learner, not a compromise

- **The XGB regime LOSES to the EBM** under both a curated sweep (0.12072) and a 120-trial Optuna search
  (0.12094 — worse than the cs0.5 global with **no** regime stage, 0.12081). On the few-thousand-row,
  ~93%-noise h16-19 subset, the EBM's **additive+pairwise+bagging is the correct regularizer**, genuinely
  the stronger small-sample learner — confirmed by optimization, not assertion.
- **The gate window 16-19 is near-optimal** (sweep): 16-19 **0.12050**, 15-19 0.12051, 17-19 0.12064,
  16-20 0.12068, 16-18 0.12092 — both edges (h16 close auction, h19 AH) carry real signal.
- **The floor is real.** Every post-floor lever died for the *right* reason: the **log-signature**
  antisymmetric path basis is null (the close edge has **no chronological-order content** — a pure
  state/level regime), the turnover-**OFI** proxy is null, and five **regime-persistence** families are
  null OOS (best −0.00002; the cleanest in-sample axis, the fast/slow vol cascade, is 87% spanned).

⇒ the two open levers are **(1) new data** (auction imbalance / GEX) and **(2) DL on the linbest residual
targeting state-space (not path) nonlinearity** — exactly what ch. 04 tests.